___
# <center>Embeddings: ementas do TJSP como números</center>
___

## Aula 26

Até aqui, todo modelo da disciplina usou **colunas**: juros, valor da causa,
câmara, comarca. Mas boa parte da informação jurídica está em **texto**:
petições, sentenças, ementas. Para um modelo usar um texto, ele precisa virar
números.

Hoje vemos dois jeitos de fazer isso:

- **contar palavras**: cada ementa vira uma lista de quantas vezes cada palavra
  aparece. É simples e funciona bem quando a resposta está escrita com as
  mesmas palavras;
- **embeddings**: uma rede neural já treinada lê a ementa e devolve 384
  números que resumem o **significado** do texto. Textos com sentido parecido
  ganham números parecidos, mesmo escritos com outras palavras.

O caminho do notebook é o mesmo dos slides:

1. transformar as ementas em números, dos dois jeitos;
2. medir a **semelhança** entre duas ementas;
3. fazer uma **busca por significado**;
4. desenhar o **mapa** das ementas;
5. **classificar** ementas com cada representação e comparar;
6. medir um **viés** do modelo.

___
<div id="indice"></div>

## Índice

- [Preparação: o pacote, o modelo e a base](#base)
- [1. Texto em números](#parte1)
- [2. Semelhança entre duas ementas](#parte2)
- [3. Busca por significado](#parte3)
- [4. O mapa das ementas](#parte4)
- [5. Classificar: contar palavras ou usar o embedding?](#parte5)
- [6. Um viés do modelo](#parte6)

___
<div id="base"></div>

# Preparação: o pacote, o modelo e a base

**Rode esta primeira célula logo ao abrir o notebook.** Ela instala o pacote
`sentence-transformers`, que sabe usar modelos de embeddings. No Colab ele
costuma já vir instalado, e a célula termina rápido.

In [ ]:
%pip install -q sentence-transformers

**Os pacotes.** Leia os comentários: cada import é uma peça que aparece mais
adiante.

In [ ]:
# pandas e numpy: tabelas e contas com vetores
import pandas as pd
import numpy as np
# plotnine: gráficos, com a mesma gramática do ggplot
from plotnine import *

# o modelo de embeddings (uma rede neural já treinada)
from sentence_transformers import SentenceTransformer

# scikit-learn: as peças que já conhecemos e duas novas
from sklearn.feature_extraction.text import TfidfVectorizer  # NOVA: conta palavras
from sklearn.manifold import TSNE                            # NOVA: achata 384 números em 2, para o mapa
from sklearn.linear_model import LogisticRegression          # a regressão logística
from sklearn.model_selection import StratifiedKFold, cross_val_score  # a validação cruzada
from sklearn.pipeline import make_pipeline                   # encadeia etapas num objeto só

**A base.** São as ementas de três buscas na jurisprudência do TJSP, que você já
viu em aulas anteriores: **plano de saúde**, **dano moral** e **criminal**. A
célula lê os três arquivos do site da disciplina e junta tudo numa tabela só,
com a coluna `tema` dizendo de qual busca veio cada ementa.

In [ ]:
partes = []
for arquivo, tema in [("plano_saude", "plano de saúde"), ("dano_moral", "dano moral"), ("criminal", "criminal")]:
    parte = pd.read_csv(f"https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/tjsp_cjsg_{arquivo}.csv")
    parte["tema"] = tema                     # de qual busca veio a ementa
    partes.append(parte)
ementas = pd.concat(partes, ignore_index=True)  # empilha as três tabelas

print(ementas.shape)
ementas["tema"].value_counts()

Uma ementa, para lembrar como elas são:

In [ ]:
print(ementas.loc[0, "assunto"])
print(ementas.loc[0, "ementa"][:600])

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. Texto em números

## 1.1 Contar palavras

O jeito mais simples: fazer uma **coluna para cada palavra** do vocabulário e,
em cada ementa, anotar quanto aquela palavra aparece. O `TfidfVectorizer` faz
isso com um ajuste: palavras que aparecem em quase todas as ementas
("apelação", "recurso") pesam pouco, e palavras raras pesam mais. Por isso o
nome: TF-IDF, frequência do termo ajustada pela raridade.

In [ ]:
contador = TfidfVectorizer(strip_accents="unicode")      # tira acentos, para "voo" e "vôo" serem iguais
palavras = contador.fit_transform(ementas["ementa"])   # uma linha por ementa, uma coluna por palavra

print("ementas e palavras:", palavras.shape)

São milhares de colunas, quase todas zero em cada ementa (uma ementa usa poucas
palavras do vocabulário inteiro). As palavras com mais peso na primeira ementa:

In [ ]:
pesos = pd.Series(palavras[0].toarray()[0], index=contador.get_feature_names_out())
pesos.sort_values(ascending=False).head(10)

## 1.2 Embeddings

Agora o modelo da aula: uma rede neural com cerca de **118 milhões de
parâmetros**, treinada por outras pessoas em textos de dezenas de línguas para
dar números parecidos a frases com o mesmo sentido. Nós só usamos a rede
pronta, sem treinar nada.

A primeira célula baixa o modelo (cerca de 470 MB, uma vez só). A segunda gera
os embeddings das 1.423 ementas, o que leva cerca de um minuto no Colab. O
método que transforma textos em embeddings se chama `encode`. Complete e rode.

In [ ]:
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
# normalize_embeddings=True deixa cada vetor com tamanho 1: assim a semelhança
# entre dois textos é só a soma dos produtos dos números (o produto escalar)
embeddings = modelo.____(ementas["ementa"].tolist(), normalize_embeddings=True, show_progress_bar=True)

print("ementas e números:", embeddings.shape)
embeddings[0][:8]                                     # os 8 primeiros números da primeira ementa

Cada ementa virou **384 números**. Nenhum deles, sozinho, quer dizer alguma
coisa ("o número 17 é o de saúde"): o que importa é o conjunto. Duas ementas
sobre o mesmo assunto têm conjuntos parecidos.

**Um limite do modelo.** Ele lê no máximo 128 pedaços de palavra (tokens) de
cada texto, o que dá umas 90 palavras. O resto da ementa é ignorado. Como as
ementas começam pelo resumo do caso, o começo carrega boa parte do sentido, mas
detalhes do fim se perdem.

**Pergunta 1.** A contagem de palavras deu milhares de colunas, quase todas
zero; o embedding deu 384 números, nenhum zero. Por que o embedding consegue
dizer que "o convênio negou o remédio" e "a operadora recusou o medicamento"
são parecidas, e a contagem de palavras não?

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. Semelhança entre duas ementas

Com os vetores de tamanho 1, a **semelhança** entre duas ementas é o produto
escalar: multiplica número a número e soma. O resultado vai de -1 a 1; quanto
mais perto de 1, mais parecido o sentido. (Esse número se chama semelhança do
cosseno.)

A função abaixo mostra o começo das duas ementas e a semelhança entre elas.

In [ ]:
def semelhanca(i, j):
    """Semelhança entre as ementas das linhas i e j, com o começo de cada uma."""
    s = embeddings[i] @ embeddings[j]               # o produto escalar
    for k in (i, j):
        print(f"[{ementas.loc[k, 'tema']} | {ementas.loc[k, 'assunto']}]", ementas.loc[k, "ementa"][:150], "...")
    print(f"semelhança: {s:.2f}")


# duas ementas de plano de saúde
semelhanca(1, 226)

Agora compare a mesma ementa de plano de saúde (linha 1) com uma criminal. As
ementas criminais começam na linha 948 (488 de plano de saúde mais 460 de dano
moral). Complete com a linha 949.

In [ ]:
semelhanca(1, ____)

Um par só pode enganar. A tabela abaixo faz a média da semelhança entre todas
as ementas de cada tema com todas as de cada outro tema.

In [ ]:
todas = embeddings @ embeddings.T                        # cada ementa com cada outra
medias = pd.DataFrame(todas).groupby(ementas["tema"].to_numpy()).mean()   # média por linha de tema
medias.T.groupby(ementas["tema"].to_numpy()).mean().round(2)              # e por coluna de tema

**Pergunta 2.** O que a diagonal da tabela mostra, comparada ao resto? Teste
outros pares (as ementas de dano moral estão nas linhas 488 a 947) e diga se
algum par de temas diferentes teve semelhança alta. Por que isso pode acontecer?

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. Busca por significado

Uma busca de jurisprudência comum procura as **palavras** digitadas. Com
embeddings, a busca transforma a consulta em 384 números e devolve as ementas
com a maior semelhança: as de **sentido** mais parecido.

As duas funções abaixo fazem as duas buscas, para comparar.

In [ ]:
def mostra(notas, k):
    """As k ementas com as maiores notas, numa tabela."""
    melhores = np.argsort(-notas)[:k]                # as posições das k maiores notas
    tabela = ementas.loc[melhores, ["tema", "assunto"]].copy()
    tabela["nota"] = notas[melhores].round(2)
    tabela["começo da ementa"] = ementas.loc[melhores, "ementa"].str[:120]
    return tabela


def busca_por_significado(consulta, k=5):
    vetor = modelo.encode([consulta], normalize_embeddings=True)[0]   # a consulta vira 384 números
    return mostra(embeddings @ vetor, k)                            # semelhança com cada ementa


def busca_por_palavra(consulta, k=5):
    vetor = contador.transform([consulta])                         # a consulta vira contagem de palavras
    return mostra((palavras @ vetor.T).toarray().ravel(), k)


busca_por_significado("o convênio não quis pagar o remédio do câncer")

In [ ]:
busca_por_palavra("o convênio não quis pagar o remédio do câncer")

**Sua vez.** Escreva uma consulta do jeito que um cliente contaria o caso, sem
termos jurídicos (por exemplo, "a loja vendeu uma geladeira quebrada e não quis
trocar"). Ponha o texto entre aspas no lugar de `____` e rode as duas buscas.

In [ ]:
consulta = ____
busca_por_significado(consulta)

In [ ]:
busca_por_palavra(consulta)

**Pergunta 3.** Qual das duas buscas trouxe resultados melhores para a sua
consulta? Tente uma consulta em que a busca por palavra ganhe, e diga o que ela
tem de diferente.

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. O mapa das ementas

Não dá para desenhar 384 dimensões. O **t-SNE** achata os 384 números em 2,
tentando manter perto, no desenho, as ementas que eram parecidas. As distâncias
grandes do mapa não valem muito; o que vale é quem está perto de quem.

In [ ]:
achatado = TSNE(n_components=2, random_state=26, init="pca").fit_transform(embeddings)
ementas["x"], ementas["y"] = achatado[:, 0], achatado[:, 1]

(ggplot(ementas, aes("x", "y", color="tema"))
 + geom_point(alpha=0.6, size=1.5)
 + labs(x="", y="", color="tema da busca", title="1.423 ementas do TJSP pelo significado")
 + theme_minimal()
 + theme(axis_text=element_blank(), figure_size=(8, 6)))

O modelo **nunca soube** de qual busca veio cada ementa: a cor foi posta
depois. Mesmo assim, os temas se separam. Para medir isso sem o desenho, a
célula abaixo pega, para cada ementa, as cinco mais parecidas (nos 384 números)
e conta quantas são do mesmo tema.

In [ ]:
semelhancas = embeddings @ embeddings.T          # a semelhança de cada ementa com cada outra
np.fill_diagonal(semelhancas, -1)                # a própria ementa não conta
vizinhas = np.argsort(-semelhancas, axis=1)[:, :5]

tema = ementas["tema"].to_numpy()
mesmo_tema = (tema[vizinhas] == tema[:, None]).mean()
print(f"das 5 vizinhas, {mesmo_tema:.0%} são do mesmo tema")

**Pergunta 4.** Olhe o mapa. Plano de saúde e dano moral ficam mais perto um do
outro do que do criminal. Por que isso faz sentido? (Dica: leia o assunto de
algumas ementas de plano de saúde.)

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. Classificar: contar palavras ou usar o embedding?

Embeddings também podem entrar num modelo como **variáveis explicativas**: as
384 colunas no lugar de juros e causa. A pergunta aqui, nas ementas de plano de
saúde e dano moral: **a decisão reconheceu dano moral?** (coluna
`tem_dano_moral`).

Vamos comparar a mesma regressão logística com as duas representações, pela
AUC na validação cruzada, como nas aulas 23 e 24.

In [ ]:
civel = (ementas["tema"] != "criminal").to_numpy()        # só plano de saúde e dano moral
y = ementas.loc[civel, "tem_dano_moral"].astype(int)
pastas = StratifiedKFold(5, shuffle=True, random_state=26)

# com a contagem de palavras: o contador entra no pipeline, para ser ajustado
# só nas partes de ajuste de cada rodada
modelo_palavras = make_pipeline(TfidfVectorizer(min_df=3), LogisticRegression(C=10, max_iter=3000))
auc_palavras = cross_val_score(modelo_palavras, ementas.loc[civel, "ementa"], y, cv=pastas, scoring="roc_auc").mean()
print("AUC contando palavras:", round(auc_palavras, 3))

Agora com os embeddings. Os 384 números de cada ementa já estão prontos em
`embeddings`; complete com as linhas das ementas cíveis.

In [ ]:
X_embeddings = embeddings[____]
auc_embeddings = cross_val_score(LogisticRegression(max_iter=3000), X_embeddings, y, cv=pastas, scoring="roc_auc").mean()
print("AUC com embeddings:", round(auc_embeddings, 3))

**Pergunta 5.** Qual representação ganhou, e por quê? Pense em como uma ementa
escreve que houve dano moral. Dê um exemplo de tarefa em que você esperaria o
contrário.

[Volta ao Índice](#indice)

___
<div id="parte6"></div>

# 6. Um viés do modelo

O modelo aprendeu com textos escritos por pessoas, e aprende também os
estereótipos desses textos. Um teste simples: comparar a frase "Essa pessoa é
engenheira." com "Ela é uma mulher." e com "Ele é um homem.". A palavra está no
feminino; se o modelo não tivesse viés, a frase deveria ficar mais perto de
"mulher".

A função abaixo faz a conta para uma lista de profissões. A coluna `diferenca`
é a semelhança com "mulher" menos a semelhança com "homem": **negativa** quer
dizer que a frase ficou mais perto de "homem".

In [ ]:
def vies(profissoes):
    referencias = modelo.encode(["Ela é uma mulher.", "Ele é um homem."], normalize_embeddings=True)
    linhas = []
    for p in profissoes:
        v = modelo.encode([f"Essa pessoa é {p}."], normalize_embeddings=True)[0]
        linhas.append({"profissão": p, "mulher": v @ referencias[0], "homem": v @ referencias[1]})
    tabela = pd.DataFrame(linhas)
    tabela["diferenca"] = tabela["mulher"] - tabela["homem"]
    return tabela.sort_values("diferenca").round(3)


vies(["enfermeira", "engenheira", "juíza", "presidente da empresa"])

**Sua vez.** Complete a lista com pelo menos quatro profissões ou cargos
jurídicos, no feminino (por exemplo, "promotora", "delegada", "estagiária",
"sócia do escritório").

In [ ]:
vies([____])

**Pergunta 6.** O que você encontrou? Imagine um sistema que usa esse modelo
para buscar currículos de "pessoa engenheira com experiência". Que risco o
resultado traz, e que cuidado você pediria antes de usar o sistema (pense na
aula 25)?

[Volta ao Índice](#indice)